# One neuron, several inputs

So far, we have trained one number at a time. In this notebook, one prediction will use several input numbers. The goal is to keep the math visible before we let any helper object do automatic gradient bookkeeping.

## Dot product: multiply matching things, then add them

A tiny neuron gives each input one matching weight. The dot product means: multiply each input by its matching weight, then add those products together. After that, add the bias.

```text
prediction = w0*x0 + w1*x1 + bias
```

## Shape table for this first example

Shape means how many numbers each thing contains. The inputs and weights both have two numbers because each input needs one matching weight.

| Thing | Value | Shape meaning |
|---|---:|---|
| `inputs` | `[4.0, 2.0]` | 2 numbers |
| `weights` | `[0.25, 3.0]` | 2 numbers |
| `bias` | `-1.0` | 1 number |
| `prediction` | one number | scalar output |

## Hand gradients for the prediction

For the prediction formula only, each weight's gradient is its matching input:

```text
d_prediction/d_weight_i = input_i
```

The bias is added directly, so its prediction gradient is always `1`:

```text
d_prediction/d_bias = 1
```

In [1]:
inputs = [4.0, 2.0]
weights = [0.25, 3.0]
bias = -1.0

product_0 = weights[0] * inputs[0]
product_1 = weights[1] * inputs[1]

prediction = product_0 + product_1 + bias

weight_grads_for_prediction = [inputs[0], inputs[1]]
bias_grad_for_prediction = 1.0

print("product_0:", product_0)
print("product_1:", product_1)
print("prediction:", prediction)
print("weight grads for prediction:", weight_grads_for_prediction)
print("bias grad for prediction:", bias_grad_for_prediction)

product_0: 1.0
product_1: 6.0
prediction: 6.0
weight grads for prediction: [4.0, 2.0]
bias grad for prediction: 1.0


## Reading the output

The two products are the two matched input-weight pairs. The prediction is the sum of those products plus the bias. The weight-gradient list has the same length as `weights` because each weight gets one matching gradient.

## Checkpoint: which input controls which weight gradient?

For this prediction:

```text
prediction = w0*x0 + w1*x1 + bias
```

the gradient for each weight is the input at the same position.

Using:

```python
inputs = [4.0, 2.0]
weights = [0.25, 3.0]
```

answer these before moving on:

1. Which input controls `d_prediction/d_weight_0`?
2. Which input controls `d_prediction/d_weight_1`?
3. Why is `d_prediction/d_bias` equal to `1`?

## Rebuild the prediction with the shared `Value` helper

Now use the `Value` helper from `src/tiny_mlp_course/value.py`. This keeps the same prediction formula, but each number can also store a gradient. Calling `prediction.backward()` asks: how does the prediction change when each earlier value changes?

In [2]:
from tiny_mlp_course.value import Value

x0 = Value(4.0)
x1 = Value(2.0)

w0 = Value(0.25)
w1 = Value(3.0)

bias = Value(-1.0)

prediction = (w0 * x0) + (w1 * x1) + bias

prediction.backward()

print("prediction:", prediction)
print("weight grads for prediction:", w0.grad, w1.grad, end="")
print("bias grad for prediction:", bias.grad)

prediction: Value(data=6.0, grad=1.0)
weight grads for prediction: 4.0 2.0bias grad for prediction: 1.0


## Compare helper gradients to the hand gradients

The helper-computed gradients should match the hand rules from above. For the prediction only, `w0.grad` should be `x0`, `w1.grad` should be `x1`, and `bias.grad` should be `1`. This is the check that the shared helper is doing the same bookkeeping we did by hand.

## Add a target and squared-error loss

The prediction gradient told us how the prediction changes when a weight changes. For training, we care about the loss. The target is the value we wanted, the error is `prediction - target`, and the squared-error loss is `error ** 2`.

The shared `Value` helper supports subtraction, so we can write `prediction - target` directly. The result is still a `Value`, which means the error step stays connected for `.backward()`.

In [3]:
from tiny_mlp_course.value import Value

x0 = Value(4.0)
x1 = Value(2.0)

w0 = Value(0.25)
w1 = Value(3.0)

bias = Value(-1.0)
target = Value(10.0)

prediction = (w0 * x0) + (w1 * x1) + bias
error = prediction - target
loss = error**2

loss.backward()

print("prediction:", prediction.data)
print("target:", target.data)
print("error:", error.data)
print("loss:", loss.data)
print("w0 grad:", w0.grad)
print("w1 grad:", w1.grad)
print("bias grad:", bias.grad)

prediction: 6.0
target: 10.0
error: -4.0
loss: 16.0
w0 grad: -32.0
w1 grad: -16.0
bias grad: -8.0


## Reading the loss-gradient output

These gradients are now loss gradients, not prediction gradients. The prediction is `6.0`, the target is `10.0`, and the error is `-4.0`, so the loss is `16.0`. Because the loss includes the error step, each weight gradient is the matching input multiplied by the loss's sensitivity to the prediction.

## Put the `Value` neuron prediction in a reusable function

The earlier `Value` example wrote `(w0 * x0) + (w1 * x1) + bias` by hand. This function keeps the same graph-building operations, but it loops over the input-weight pairs so the code can handle any matching list length.

The important detail is that `weighted_sum` starts as `Value(0.0)` and each step uses `input_value * weight`. That keeps the operation history connected, so calling `.backward()` on the prediction can still fill in each weight's gradient.

In [23]:
from tiny_mlp_course.value import Value

def predict_one_neuron(
    inputs: list[Value],
    weights: list[Value],
    bias: Value,
) -> Value:
    if len(inputs) != len(weights):
        raise ValueError("inputs and weights must have the same length")

    weighted_sum = Value(0.0)

    for input_value, weight in zip(inputs, weights):
        weighted_sum = weighted_sum + input_value * weight

    return weighted_sum + bias


x0 = Value(4.0)
x1 = Value(2.0)
w0 = Value(0.25)
w1 = Value(3.0)
bias = Value(-1.0)

prediction = predict_one_neuron(
    inputs=[x0, x1],
    weights=[w0, w1],
    bias=bias,
)

prediction.backward()

print("prediction:", prediction)
print("weight grads for prediction:", w0.grad, w1.grad)
print("bias grad for prediction:", bias.grad)

prediction: Value(data=6.0, grad=1.0)
weight grads for prediction: 4.0 2.0
bias grad for prediction: 1.0


## Reading the reusable helper output

The prediction is still `6.0`, so the function matches the hand-written formula. The gradients also match the earlier prediction-gradient rule: `w0.grad` is `4.0`, `w1.grad` is `2.0`, and `bias.grad` is `1.0`.

This check tells us the helper did not break the teaching graph. We can now reuse the same function in later cells when the notebook moves from prediction gradients to loss gradients and weight updates.